# Notebook for Post Processing

README:
- select the venv as kernel --> if havent made refer to ReadMe
- create/start virtual env (if havent alr)
- ensure these installations are are done

You should have created an anaconda virtual environment with python by this time. If not, try the setup section in the readme? 

## Setting up

Try running the cell below to check what environment this is currently running in. If the name printed is the correct environment name, the setup is done! 

In [ ]:
import sys
import os
import pandas as pd

#will print the name of the currently active environment 
kernel_name = os.path.basename(sys.executable.replace("/bin/python",""))
print(kernel_name)

*Note to recheck files created after each run 

Make sure requirements.txt is in the same directory as this file. 

Run the pip install command below to double-check all requirements are installed and found here too. 
* You may need to resart the kernel after. 


> <i> YOU MIGHT NEED TO RUN THE CELL BELOW EVERYTIME USING THIS SCRIPT </i>

In [ ]:
pip install -r requirements.txt

## Organising folders, renaming files 

  1. The first step after obtaining CSV files of XY coordinates from Deeplabcut is to rename the files.
     * Deeplabcut adds a large suffix to the end of each CSV file created
       (e.g DLC_resnet50_KF_Full...le1_50000_el_filtered.csv)
     * To avoid issues caused by too long file names remove the added suffix:
       change _d48_23_F1_FDLC_resnet50_KF_Full...le1_50000_el_filtered.csv_
       to _d48_23_F1.csv_
    
  2. Once the files have been renamed create the following folder hierarchy:
    - First created a folder called Raw_CSVs.
    - Within the Raw_CSVs folder create a folder titled the name of the batch you want to analyse (e.g d48) and place all the raw CSV files containing the XY coordinates inside.
    - Within the batch folder create a folder called Cropped_CSV.
        - The Cropped_CSV folder is where the output of SUBSET_CSV.py and Interpolate_convert_mm.py will be written.

In [ ]:
import os
import shutil
import re

# SET THIS to your folder with DLC CSVs (might be in the same folder as where the videos analysed for DLC are) 
source_folder = "/Users/teeshabasak/Desktop/DLC/20mar2025-teesha-2025-03-20/videos_for_analysis" 
batch_name = "expMar2025"  # e.g. "d48", "day1", or any label you want for this batch

# Create folder structure for saving outputs
current_dir = os.getcwd()
raw_csvs_folder = os.path.join(current_dir, "Raw_CSVs")
target_folder = os.path.join(raw_csvs_folder, batch_name)
cropped_folder = os.path.join(target_folder, "Cropped_CSV")
target_folder = os.path.join("Raw_CSVs", batch_name)
cropped_folder = os.path.join(target_folder, "Cropped_CSV")

os.makedirs(cropped_folder, exist_ok=True)
print(f"Created folders: {target_folder} and {cropped_folder}")

In [ ]:
# Rename & move files
for filename in os.listdir(source_folder):
    if filename.endswith(".csv"):
        # Match pattern like: d48_14_FDLC_Resnet50... and keep only: d48_14_F.csv
        match = re.match(r'^(.+?_[A-Z])DLC.*\.csv$', filename)
        if match:
            simplified_name = match.group(1) + ".csv"

            old_path = os.path.join(source_folder, filename)
            new_path = os.path.join(target_folder, simplified_name)

            shutil.copy2(old_path, new_path)
            print(f"Copied and renamed: {filename} → {simplified_name}")
        else:
            print(f"Skipped (no match): {filename}")

In [ ]:
#Set this to the folder where your videos are currently stored
source_folder = "/Users/teeshabasak/Desktop/DLC/20mar2025-teesha-2025-03-20/videos_for_analysis"

# Destination folder
dest_folder = os.path.join(os.getcwd(), "data", "videos")
os.makedirs(dest_folder, exist_ok=True)

# Copy only unlabelled .mp4 files (exclude those with "DLC" or "labeled" in the name)
copied_files = []
for filename in os.listdir(source_folder):
    if filename.endswith(".mp4") and "DLC" not in filename and "labeled" not in filename:
        src_path = os.path.join(source_folder, filename)
        dst_path = os.path.join(dest_folder, filename)
        shutil.copy2(src_path, dst_path)
        copied_files.append(filename)

print(f"Copied {len(copied_files)} unlabelled .mp4 files to {dest_folder}")
if copied_files:
    print("Files copied:")
    for f in copied_files:
        print(" •", f)
else:
    print("No matching unlabelled .mp4 files found.")

## Run: LED_times.py

<b>Purpose:</b> finds the exact frames where the LED light turns on and off in each video.  
- **Input**: folder of videos (.mp4).  
- **Output**: CSV file (`LED_times.csv`) that records the start and end time of the LED light in each video.


NOTE: `--no_hist_thresh` disables the automatic adjustment of the LED brightness threshold based on the video’s brightness.
* When this flag is <u>not used</u>, the script looks at the brightness histogram of the video to pick a better threshold for detecting the LED.
* When this flag is <u>used</u>, the script uses fixed default thresholds (35 for normal, 25 for sensitive) instead of adapting to each video.
<b> Use this flag only if the histogram method is not working well for your videos. </b>


In [ ]:
%run scripts/LED_times.py \
  --video_path data/videos/ \
  --output_path data/output/ \
  --no_hist_thresh \
  --max_led 1

### Better to run this on a computer with GPU (toothless) 
<i> Will be much faster (will take a while anyway) </i> 

<u> STEPS: </u> 
1. Activate virtual environment
2. Navigate into project directory (will refer to it as `mainDir`)
3. Can transfer the whole project directory into toothless OR
   make sure of the following:
   * requirements.txt is in the main project dir
   * LED_times.py is inside mainDir/scripts
   * all the videos are inside mainDir/data/videos
5. Run: `pip install -r requirements.txt` to ensure all libraries/packages are installed
7. Run: `python scripts/LED_times.py --video_path data/videos/ --output_path data/output/ --max_led 1`
8. Check mainDir/data/... for output 


## Run: SUBSET_CSV_M.py

<i> Only working with CSVs - can be run on Mac, since very lightweight process. </i> 

<b>Purpose:</b> uses the LED timings and the original tracking CSVs to cut out specific time windows (before, during, and after the LED light) for each video.  
- **Input**: The tracking CSV files and the LED_times CSV file.  
- **Output**: New CSV files for each video that only contain data from selected time periods, saved in a folder called `subset`.


NOTE: Before running this, ensure outputs from running the LED_time.py has been moved inside the `data/output` folder.

<u>Regarding the subsets chosen:</u>

The values seen below were chosen to break the video data into meaningful phases around the LED event:

* `(-6, 0)`: Baseline period — 6 seconds before LED turns on - captures baseline behaviour before the LED is turned on. This helps us understand the fish's natural behaviour before any stimulus.
* `(0, 6)`: Stimulus + Recovery — LED start to 6s after - Shows immediate and longer reactions during and a little after LED.
* `(-3, 'start')`: Anticipation - 3s before to LED start - Short anticipation window right before stimulus.
* `('start','end' )`: LED on to LED off - The exact period of stimulation — captures how fish behave during LED.
* `('end', 3)`: Post-stimulus window — LED off to 3s after - Immediate post-stimulus behavior.

In [ ]:
subsets = "[(-6, 0), (0, 6), (-3, 'start'), ('start', 'end'), ('end', 3)]"

command = f"""
%run scripts/SUBSET_CSV_M.py \
  --csv_path Raw_CSVs/{batch_name} \
  --led_path data/output/LED_times.csv \
  --output_path data/output/subset/{batch_name} \
  --subsets "{subsets}"
"""

get_ipython().run_line_magic("run", command.strip().replace("%run ", ""))

# Run: data_mm_analysis.py 
#### This also runs `analysis.py` and `data_integrity.py`

<b> Analyse Behaviour by extracting:  — Distance, Velocity, Acceleration & Trajectory </b>

`data_mm_analyse.py`: performs a **full analysis of fish movement** using the subset CSVs.


- `data_integrity.py`: Cleans up and interpolates the data. If tracking was lost (e.g., fish was occluded), this fills in missing data so the analysis is smoother.
- `analysis.py`: Calculates important behavioral metrics like:
  - `Distance moved`
  - `Velocity` 
  - `Acceleration` 
  - `Angle`(trajectory direction)
  - `Distance from LED` (proximity to stimulus)
- `data_mm_analyse.py`: Runs the full workflow above and converts pixel coordinates to **millimeters**

**Inputs:**
- Subset CSVs created earlier - DETAILS ABOUT WHICH SUBSETS USED AND WHY (_0 and _1) 
- The LED_times.csv file - INTERMEDIATE STEP 

**Outputs:**
- New CSVs in an "analysed" folder that contain the original data **and movement metrics**

These analysed files are now ready for plotting, group comparisons, or statistical analysis.

#### Intermediate Step: Prepare LED Timings for Each Subset Video (before/during LED) 

Each original fish video (like `d48_01_F.csv`) has just **one LED timing entry** in `LED_times.csv`.

However, the subsets using split each video and its data into **multiple smaller files**, corresponding to each specified time segment:

- `d48_01_F_0.csv` → e.g., before the LED (6 seconds before LED to start of LED)
- `d48_01_F_1.csv` → e.g., during the LED (start of LED to 6 seconds after) 
- etc.

The following cell creates a new LED file, called `LED_times_expanded.csv`, which gives each subset file its own LED timing row (matching the naming as well).

---

<i> This step converts our original LED_times.csv (which has 1 entry of LED info per video) into a new version that includes each subset file separately (e.g., _0.csv and _1.csv).</i>

<i> This is necessary because the next script, data_mm_analyse.py, expects the LED timing to match exactly with the filenames of the CSVs being analyzed. </i>



In [ ]:
import pandas as pd
from pathlib import Path

batch_name = "expMar2025"
subset_dir = Path(f"data/output/subset/{batch_name}")
led_path = Path("data/output/LED_times.csv")
output_csv_path = Path("data/output/LED_times_expanded.csv")

# 1. Load the original LED times file
led_df = pd.read_csv(led_path)
led_df["base_name"] = led_df["name"].str.replace(".mp4", "", regex=False)

print("Unique base names from LED_times.csv:")
print(led_df["base_name"].unique()[:10])

# 2. Track seen base_names to avoid duplicate entry creation
seen_base_names = set()
subset_entries = []

for subset_file in sorted(subset_dir.glob("**/*.csv")):
    subset_name = subset_file.stem  # e.g., d48_01_F_0

    if "-checkpoint" in subset_name:
        continue

    # Extract base name: remove _0 or _1 from subset name
    if subset_name.endswith("_0") or subset_name.endswith("_1"):
        base_name = "_".join(subset_name.split("_")[:-1])
    else:
        print(f"⚠️ Could not extract base name from {subset_name}")
        continue

    if base_name in seen_base_names:
        continue  # Skip if already processed

    print(f"\nNow checking base video: {base_name}")
    match = led_df[led_df["base_name"] == base_name]

    if not match.empty:
        row = match.iloc[0].copy()
        for suffix in ["_0", "_1"]:
            new_row = row.copy()
            new_row["subset_name"] = base_name + suffix
            subset_entries.append(new_row)
            print(f"Matched and added row for {new_row['subset_name']}")
        seen_base_names.add(base_name)
    else:
        print(f"❌ No match found in LED_times.csv for: {base_name}")

# 3. Save output
if subset_entries:
    expanded_df = pd.DataFrame(subset_entries)
    expanded_df["name"] = expanded_df["subset_name"]
    expanded_df.to_csv(output_csv_path, index=False)
    print(f"\n✅ Saved expanded LED data to: {output_csv_path}")
else:
    print("\n⚠️ No rows to save. Double-check file names and 'name' column in LED_times.csv.\n This could take a few seconds...")

In [1]:
import pandas as pd
from pathlib import Path
import shutil

batch_name = "expMar2025"
subset_dir = Path(f"data/output/subset/{batch_name}")
led_csv_path = Path("data/output/LED_times_expanded.csv")

before_out = Path(f"data/output/analysed/{batch_name}/before_LED")
led_out = Path(f"data/output/analysed/{batch_name}/LED")
temp_input = Path("temp_input")

before_out.mkdir(parents=True, exist_ok=True)
led_out.mkdir(parents=True, exist_ok=True)
temp_input.mkdir(parents=True, exist_ok=True)

# Load LED metadata
led_df = pd.read_csv(led_csv_path)

# Load all subset files into a dictionary for quick lookup
all_subset_files = list(subset_dir.glob("**/*.csv"))
subset_lookup = {f.stem: f for f in all_subset_files}

print(f"🔍 Indexed {len(all_subset_files)} subset files.")

# Loop through LED-expanded metadata
for _, row in led_df.iterrows():
    subset_name = row["subset_name"]

    if subset_name not in subset_lookup:
        print(f"❌ File not found: {subset_name} in subset directory")
        continue

    full_csv_path = subset_lookup[subset_name]

    # Set output directory
    output_path = before_out if subset_name.endswith("_0") else led_out

    # Create temp folder and copy file
    temp_dir = temp_input / subset_name
    temp_dir.mkdir(parents=True, exist_ok=True)
    temp_file = temp_dir / f"{subset_name}.csv"
    shutil.copy(full_csv_path, temp_file)

    print(f"\n🚀 Running analysis for {subset_name}")
    print(f"📄 Temp CSV: {temp_file}")
    print(f"📂 Output to: {output_path}")

    # Run script
    command = f"""
    %run scripts/data_mm_analyse.py \
      --csv_path "{temp_dir}" \
      --LED_csv_path "{led_csv_path}" \
      --output_path "{output_path}"
    """
    get_ipython().run_line_magic("run", command.strip().replace("%run ", ""))

    # Clean up
    shutil.rmtree(temp_dir, ignore_errors=True)


🔍 Indexed 252 subset files.

🚀 Running analysis for d48_01_F_0
📄 Temp CSV: temp_input/d48_01_F_0/d48_01_F_0.csv
📂 Output to: data/output/analysed/expMar2025/before_LED
1 csv files found
d48_01_F_0.csv
0.00% missing data, 20.00% low likelihood data
⚠️ Warning: d48_01_F_0.csv has 0.00% missing data
16.05% of frames are jittery
Saved to ./data/output/analysed/expMar2025/before_LED/d48_01_F_0.csv
Done
1  files with problems
Saving problems_csv.txt to output_path

🚀 Running analysis for d48_01_F_1
📄 Temp CSV: temp_input/d48_01_F_1/d48_01_F_1.csv
📂 Output to: data/output/analysed/expMar2025/LED
1 csv files found
d48_01_F_1.csv
0.00% missing data, 11.89% low likelihood data
16.11% of frames are jittery
Saved to ./data/output/analysed/expMar2025/LED/d48_01_F_1.csv
Done
0  files with problems
Saving problems_csv.txt to output_path

🚀 Running analysis for d48_01_T_0
📄 Temp CSV: temp_input/d48_01_T_0/d48_01_T_0.csv
📂 Output to: data/output/analysed/expMar2025/before_LED
1 csv files found
d48_01_T